# Quick Start: Interrupted Time Series with CausalPy

This notebook runs a complete ITS analysis in under 2 minutes.
Copy-paste this as a starting template for your own ITS study.

**What you need:**
- A time series of outcome measurements
- A known intervention date
- At least 10 pre-intervention observations

**What you get:**
- Posterior estimates for level change and slope change at the intervention
- Counterfactual trajectory with 94% HDI
- Convergence diagnostics

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import causalpy as cp
import arviz as az

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

## Step 1: Prepare Your Data

Replace this synthetic data with your actual dataset.
Required columns: a time index, the outcome variable, `treated` (0/1), and `t_post` (time since intervention, 0 before).

In [ ]:
# --- REPLACE THIS SECTION WITH YOUR DATA ---
# Synthetic example: monthly hospital admissions with a policy intervention at month 24

n_months = 48
n_pre = 24  # Intervention at month 24

months = np.arange(n_months).astype(float)
treated = (months >= n_pre).astype(float)
t_post = np.maximum(months - n_pre, 0).astype(float)

# True effects (known because this is synthetic)
TRUE_LEVEL_CHANGE = -10.0  # Immediate drop
TRUE_SLOPE_CHANGE = -0.2   # Additional decline per month post-intervention

outcome = (
    80.0                            # Baseline
    - 0.2 * months                  # Pre-trend (secular decline)
    + TRUE_LEVEL_CHANGE * treated   # Level change
    + TRUE_SLOPE_CHANGE * t_post * treated  # Slope change
    + np.random.normal(0, 3.5, n_months)    # Noise
)

df = pd.DataFrame({
    "month": months,
    "outcome": outcome,
    "treated": treated,
    "t_post": t_post,
})

print(f"Dataset: {len(df)} months, intervention at month {n_pre}")
print(f"Pre-period mean:  {df.loc[df['treated']==0, 'outcome'].mean():.1f}")
print(f"Post-period mean: {df.loc[df['treated']==1, 'outcome'].mean():.1f}")
print(df.head(3))

## Step 2: Fit the ITS Model

The formula `outcome ~ 1 + month + treated + t_post` fits the standard ITS model:
- `1`: intercept
- `month`: pre-intervention trend
- `treated`: level change at intervention
- `t_post`: slope change (additional monthly change post-intervention)

In [ ]:
model = cp.InterruptedTimeSeries(
    data=df,
    treatment_time=n_pre,
    formula="outcome ~ 1 + month + treated + t_post",
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={
            "draws": 1000,
            "tune": 1000,
            "chains": 4,
            "target_accept": 0.90,
            "random_seed": 42,
        }
    ),
)

print("Model fitted.")

## Step 3: Check Convergence

In [ ]:
summary = az.summary(model.idata, var_names=["treated", "t_post", "month", "sigma"],
                     round_to=3)
print("Convergence Summary")
print(summary[["mean", "sd", "hdi_3%", "hdi_97%", "r_hat", "ess_bulk"]].to_string())
print()

max_rhat = summary["r_hat"].max()
min_ess = summary["ess_bulk"].min()
print(f"Max R-hat:   {max_rhat:.4f}  (pass: < 1.01)")
print(f"Min ESS:     {min_ess:.0f}     (pass: > 400)")

if max_rhat < 1.01 and min_ess > 400:
    print("CONVERGED: All diagnostics pass.")
else:
    print("WARNING: Check convergence. Increase tune/draws or target_accept.")

## Step 4: Report the Causal Effects

In [ ]:
# Extract posterior for causal parameters
treated_samples = model.idata.posterior["treated"].values.flatten()
t_post_samples = model.idata.posterior["t_post"].values.flatten()

level_mean = treated_samples.mean()
level_hdi = az.hdi(treated_samples, hdi_prob=0.94)
level_prob_neg = (treated_samples < 0).mean()

slope_mean = t_post_samples.mean()
slope_hdi = az.hdi(t_post_samples, hdi_prob=0.94)
slope_prob_neg = (t_post_samples < 0).mean()

print("Causal Effect Estimates")
print("=" * 55)
print(f"{'Parameter':<20} {'Mean':>8} {'94% HDI':>22} {'P(<0)':>8}")
print("-" * 55)
print(f"  Level change     {level_mean:>+8.2f} "
      f"[{level_hdi[0]:>+8.2f}, {level_hdi[1]:>+8.2f}] {level_prob_neg:>7.1%}")
print(f"  Slope change     {slope_mean:>+8.2f} "
      f"[{slope_hdi[0]:>+8.2f}, {slope_hdi[1]:>+8.2f}] {slope_prob_neg:>7.1%}")
print("-" * 55)
print(f"  True level:      {TRUE_LEVEL_CHANGE:>+8.2f}")
print(f"  True slope:      {TRUE_SLOPE_CHANGE:>+8.2f}")

# Cumulative effect over post-intervention period
n_post = n_months - n_pre
cumulative_samples = sum(
    treated_samples + t_post_samples * t
    for t in range(n_post)
)
cum_hdi = az.hdi(cumulative_samples, hdi_prob=0.94)
print()
print(f"  Cumulative effect ({n_post} months): {cumulative_samples.mean():>+8.1f} "
      f"[{cum_hdi[0]:>+8.1f}, {cum_hdi[1]:>+8.1f}]")

## Step 5: Visualize

In [ ]:
# Use CausalPy's built-in plot
fig, axes = model.plot()
fig.suptitle("ITS: Observed vs Counterfactual with 94% HDI", fontsize=13)
plt.tight_layout()
plt.show()

print()
print("Interpretation:")
print("  Solid line = observed outcome")
print("  Dashed line + shaded band = counterfactual (what would have happened without intervention)")
print("  Gap at/after intervention = estimated causal effect")

## Quick Reference: Formula Variants

| Formula | Model Type | When to Use |
|---------|-----------|-------------|
| `y ~ 1 + t + treated + t_post` | Level + slope change | Default |
| `y ~ 1 + t + treated` | Level change only | Short post-period |
| `y ~ 1 + t + t_post` | Slope change only | Gradual implementation |
| `y ~ 1 + t + treated + t_post + C(month)` | With seasonality | Monthly seasonality |

## Next Steps

- For seasonal data: add `C(calendar_month)` or Fourier terms
- For model comparison: use LOO with `az.compare()`
- For formal inference: run posterior predictive checks (Module 02, Notebook 03)
- For panel data: see `quickstart_synthetic_control.ipynb`